# 🏥 Dataset 2: MIMIC-III (Demo)
**Propósito:** Análisis clínico — Predicción de mortalidad hospitalaria en UCI.  
**Fuente:** MIT Lab for Computational Physiology — Beth Israel Deaconess Medical Center (2001–2012).  
> ⚠️ Esta es la versión DEMO (100 pacientes). El dataset completo requiere registro en PhysioNet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
BASE = '../datasets/5 data sets/dataset2/'
print("✅ Librerías cargadas")

## 1. Carga de Tablas Relacionales

In [ ]:
patients    = pd.read_csv(BASE + 'PATIENTS.csv')
admissions  = pd.read_csv(BASE + 'ADMISSIONS.csv')
icustays    = pd.read_csv(BASE + 'ICUSTAYS.csv')
labevents   = pd.read_csv(BASE + 'LABEVENTS.csv')
prescriptions = pd.read_csv(BASE + 'PRESCRIPTIONS.csv')

print(f"PATIENTS    : {patients.shape}")
print(f"ADMISSIONS  : {admissions.shape}")
print(f"ICUSTAYS    : {icustays.shape}")
print(f"LABEVENTS   : {labevents.shape}")
print(f"PRESCRIPTIONS: {prescriptions.shape}")

## 2. Construcción de la Tabla Maestra (Join Relacional)

In [ ]:
# Unir PATIENTS + ADMISSIONS
df = admissions.merge(patients[['SUBJECT_ID','GENDER','DOB','DOD']], 
                      on='SUBJECT_ID', how='left')

# Calcular edad al momento de admisión
df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'])
df['DOB']       = pd.to_datetime(df['DOB'])
df['DOD']       = pd.to_datetime(df['DOD'])
df['AGE'] = ((df['ADMITTIME'] - df['DOB']).dt.days / 365.25).round(1)
# MIMIC censura edades >89 con valores artificiales (~300 años)
df['AGE'] = df['AGE'].clip(upper=89)

# Variable objetivo: HOSPITAL_EXPIRE_FLAG (1=murió en hospital, 0=sobrevivió)
df['TARGET'] = df['HOSPITAL_EXPIRE_FLAG'].astype(int)

# Añadir duración de estancia en UCI
icu_dur = icustays.groupby('HADM_ID')['LOS'].sum().reset_index()
icu_dur.columns = ['HADM_ID','ICU_LOS']
df = df.merge(icu_dur, on='HADM_ID', how='left')

print(f"Shape tabla maestra: {df.shape}")
print(f"Mortalidad hospitalaria: {df['TARGET'].mean():.1%}")
df[['SUBJECT_ID','AGE','GENDER','ADMISSION_TYPE','ICU_LOS','TARGET']].head(5)

## 3. Feature Engineering — Conteos de Laboratorio y Medicamentos

In [ ]:
# Número de eventos de laboratorio por admisión
lab_counts = labevents.groupby('HADM_ID').size().reset_index(name='N_LAB_EVENTS')
presc_counts = prescriptions.groupby('HADM_ID').size().reset_index(name='N_PRESCRIPTIONS')

df = df.merge(lab_counts, on='HADM_ID', how='left')
df = df.merge(presc_counts, on='HADM_ID', how='left')
df[['N_LAB_EVENTS','N_PRESCRIPTIONS']] = df[['N_LAB_EVENTS','N_PRESCRIPTIONS']].fillna(0)
print("Features de conteo añadidas ✅")

## 4. Análisis Exploratorio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Mortalidad
axes[0].bar(['Sobrevivió','Murió'], df['TARGET'].value_counts().sort_index(),
            color=['steelblue','tomato'])
axes[0].set_title('Distribución de Mortalidad Hospitalaria')

# Edad
df[df['AGE']>0]['AGE'].hist(ax=axes[1], bins=20, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de Edad')
axes[1].set_xlabel('Años')

# LOS UCI
df['ICU_LOS'].clip(0,30).hist(ax=axes[2], bins=20, color='seagreen', edgecolor='white')
axes[2].set_title('Duración en UCI (días, cap=30)')
axes[2].set_xlabel('Días')

plt.tight_layout()
plt.savefig('mimic_eda.png', dpi=100)
plt.show()

## 5. Limpieza y Preparación

In [ ]:
# Codificar género
df['GENDER_N'] = (df['GENDER'] == 'M').astype(int)

# Codificar tipo de admisión
adm_map = {'EMERGENCY':0,'URGENT':1,'ELECTIVE':2,'NEWBORN':3}
df['ADMISSION_TYPE_N'] = df['ADMISSION_TYPE'].map(adm_map).fillna(0).astype(int)

# Features finales
features = ['AGE','GENDER_N','ADMISSION_TYPE_N','ICU_LOS','N_LAB_EVENTS','N_PRESCRIPTIONS']
df_model = df[features + ['TARGET']].dropna()
print(f"Registros para modelado: {len(df_model)}")

X = df_model[features]
y = df_model['TARGET']

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)}  Test: {len(X_test)}")

## 6. Modelo — Regresión Logística
*(Clasificación supervisada binaria — Mortalidad hospitalaria)*

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:,1]

print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, target_names=['Sobrevivió','Murió']))
try:
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
except:
    print("ROC-AUC: no calculable (pocas muestras por clase)")

# Coeficientes
coef_df = pd.DataFrame({'Feature': features, 'Coeficiente': lr.coef_[0]})
print("\n=== COEFICIENTES (Función Sigmoide) ===")
print(coef_df.sort_values('Coeficiente', ascending=False).to_string(index=False))

In [ ]:
# Visualización de coeficientes
coef_df.set_index('Feature').sort_values('Coeficiente').plot(
    kind='barh', figsize=(8,4), color=['tomato' if x<0 else 'steelblue' 
    for x in coef_df.set_index('Feature').sort_values('Coeficiente')['Coeficiente']])
plt.title('Coeficientes de Regresión Logística\n(mayor coef = mayor riesgo de mortalidad)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('mimic_coef.png', dpi=100)
plt.show()

## 7. La Función Sigmoide
> La regresión logística usa la función sigmoide para convertir la salida lineal en una **probabilidad** entre 0 y 1:
> $$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8,4))
plt.plot(z, sigmoid, 'steelblue', linewidth=2.5)
plt.axhline(0.5, color='tomato', linestyle='--', label='Umbral de decisión (0.5)')
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid > 0.5), alpha=0.1, color='green', label='Predice MUERTO (1)')
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5), alpha=0.1, color='red', label='Predice SOBREVIVE (0)')
plt.title('Función Sigmoide — Regresión Logística')
plt.xlabel('z (combinación lineal de features)')
plt.ylabel('P(mortalidad = 1)')
plt.legend()
plt.tight_layout()
plt.savefig('mimic_sigmoide.png', dpi=100)
plt.show()

## ✅ Resumen
| Tabla | Registros |
|---|---|
| PATIENTS | 100 |
| ADMISSIONS | 129 |
| ICUSTAYS | 136 |
| LABEVENTS | ~260K |

**Notas importantes:**
- Dataset demo: resultados no generalizables al dataset completo
- La Regresión Logística produce probabilidades directamente interpretables
- ROC-AUC es la métrica correcta para datos clínicos desbalanceados
- Edades >89 años fueron censuradas en el dataset original por anonimato